In [2]:
import os
import gradio as gr

# --- POWDER COATING COMPENSATION ---
POWDER_COAT_ALLOWANCE = 0.3

# CORRECTED MIL-SPEC RIGID PALS GEOMETRY (Post-Coating Target Dimensions)
TARGET_SLOT_WIDTH = 25.4       # 1.0" target slot width
TARGET_SLOT_HEIGHT = 12.7      # 0.5" target slot height
COL_PITCH = 38.1               # Exact 1.5" center-to-center column spacing
WEB_BRIDGE = 25.4              # Exact 1.0" solid metal horizontal webbing face
ROW_PITCH = WEB_BRIDGE + TARGET_SLOT_HEIGHT  # 38.1 mm center-to-center row spacing

# Dynamic cutting dimensions used by the laser cutter
SLOT_WIDTH = TARGET_SLOT_WIDTH + POWDER_COAT_ALLOWANCE
SLOT_HEIGHT = TARGET_SLOT_HEIGHT + POWDER_COAT_ALLOWANCE
HANDLE_HEIGHT = 25.4 + POWDER_COAT_ALLOWANCE # 1 inch standard handle slot

def generate_molle_svg(panel_width, panel_height, margin, paracord_dia, mount_hole_dia, corner_chamfer, softness,
                       handle_top, handle_bottom, handle_left, handle_right, include_mounts):

    # 1. Calculate clearances for all four sides based on active handles
    # We add half a margin of breathing room between the handle and the grid
    clear_t = margin + (HANDLE_HEIGHT + margin / 2.0 if handle_top else 0)
    clear_b = margin + (HANDLE_HEIGHT + margin / 2.0 if handle_bottom else 0)
    clear_l = margin + (HANDLE_HEIGHT + margin / 2.0 if handle_left else 0)
    clear_r = margin + (HANDLE_HEIGHT + margin / 2.0 if handle_right else 0)

    # 2. Calculate usable internal area canvas
    usable_w = panel_width - clear_l - clear_r
    usable_h = panel_height - clear_t - clear_b

    # 3. Calculate exact max intervals that mathematically fit the bounding area using the cut sizes
    num_cols = int((usable_w - SLOT_WIDTH) // COL_PITCH) + 1
    num_rows = int((usable_h - SLOT_HEIGHT) // ROW_PITCH) + 1

    if num_cols < 1 or num_rows < 1:
        return "<h3 style='color:red;'>Error: Dimensions, margins, or handles too large to fit a grid.</h3>"

    # Symmetrically center the geometric grid within the available asymmetrical dimensions
    grid_w = ((num_cols - 1) * COL_PITCH) + SLOT_WIDTH
    grid_h = ((num_rows - 1) * ROW_PITCH) + SLOT_HEIGHT

    start_x = clear_l + (usable_w - grid_w) / 2.0
    start_y = clear_t + (usable_h - grid_h) / 2.0

    svg_lines = []

    # SVG Header
    svg_lines.append(f'<svg xmlns="http://www.w3.org/2000/svg" width="100%" height="100%" viewBox="0 0 {panel_width} {panel_height}">')

    # --- Calculate Soft Chamfered Polygon Points ---
    # To get perfectly rounded chamfers, we inset the polygon points by the 'softness' radius,
    # and then apply a stroke of width (softness*2) with a rounded line join.
    ins = softness
    x0, y0 = ins, ins
    x1, y1 = panel_width - ins, panel_height - ins

    # Ensure the effective chamfer doesn't invert if softness is set very high
    eff_c = max(0, min(corner_chamfer - ins, (x1-x0)/2.0, (y1-y0)/2.0))

    # 8 points defining the inset boundary with 45-degree chamfered corners
    poly_points = f"{x0+eff_c},{y0} {x1-eff_c},{y0} {x1},{y0+eff_c} {x1},{y1-eff_c} {x1-eff_c},{y1} {x0+eff_c},{y1} {x0},{y1-eff_c} {x0},{y0+eff_c}"

    # 4. Define the SVG Mask to punch transparent holes
    svg_lines.append('  <defs>')
    svg_lines.append('    <mask id="cutout-mask">')

    # The white polygon defines the solid, visible area of the panel (Inflated by the stroke to reach true dimensions)
    stroke_w = ins * 2.0
    svg_lines.append(f'      <polygon points="{poly_points}" fill="white" stroke="white" stroke-width="{stroke_w}" stroke-linejoin="round" />')

    # 5. Dynamic Handle Cutouts (Drawn in black to create transparent holes)
    handle_radius = HANDLE_HEIGHT / 2.0 # Creates a perfect pill shape

    # Horizontal Handles (Top/Bottom)
    max_horiz_handle_w = max(10.0, min(150.0, panel_width - clear_l - clear_r + (margin if not handle_left and not handle_right else 0)))
    if handle_top:
        hx = clear_l + (usable_w - max_horiz_handle_w) / 2.0
        svg_lines.append(f'      <rect x="{hx:.3f}" y="{margin:.3f}" width="{max_horiz_handle_w:.3f}" height="{HANDLE_HEIGHT:.3f}" rx="{handle_radius:.3f}" ry="{handle_radius:.3f}" fill="black" />')
    if handle_bottom:
        hx = clear_l + (usable_w - max_horiz_handle_w) / 2.0
        by = panel_height - margin - HANDLE_HEIGHT
        svg_lines.append(f'      <rect x="{hx:.3f}" y="{by:.3f}" width="{max_horiz_handle_w:.3f}" height="{HANDLE_HEIGHT:.3f}" rx="{handle_radius:.3f}" ry="{handle_radius:.3f}" fill="black" />')

    # Vertical Handles (Left/Right)
    max_vert_handle_h = max(10.0, min(150.0, panel_height - clear_t - clear_b + (margin if not handle_top and not handle_bottom else 0)))
    if handle_left:
        vy = clear_t + (usable_h - max_vert_handle_h) / 2.0
        svg_lines.append(f'      <rect x="{margin:.3f}" y="{vy:.3f}" width="{HANDLE_HEIGHT:.3f}" height="{max_vert_handle_h:.3f}" rx="{handle_radius:.3f}" ry="{handle_radius:.3f}" fill="black" />')
    if handle_right:
        rx = panel_width - margin - HANDLE_HEIGHT
        vy = clear_t + (usable_h - max_vert_handle_h) / 2.0
        svg_lines.append(f'      <rect x="{rx:.3f}" y="{vy:.3f}" width="{HANDLE_HEIGHT:.3f}" height="{max_vert_handle_h:.3f}" rx="{handle_radius:.3f}" ry="{handle_radius:.3f}" fill="black" />')


    # 6. Main Mil-Spec Grid Generation
    for r in range(num_rows):
        y = start_y + (r * ROW_PITCH)
        for c_col in range(num_cols):
            x = start_x + (c_col * COL_PITCH)

            # Create the actual strap slot
            svg_lines.append(f'      <rect x="{x:.3f}" y="{y:.3f}" width="{SLOT_WIDTH:.3f}" height="{SLOT_HEIGHT:.3f}" rx="2.0" ry="2.0" fill="black" />')

            # Utility Paracord Openings
            if r < num_rows - 1 and c_col < num_cols - 1:
                px = x + SLOT_WIDTH + ((COL_PITCH - SLOT_WIDTH) / 2.0)
                py = y + SLOT_HEIGHT + (WEB_BRIDGE / 2.0)
                cut_paracord_radius = (paracord_dia + POWDER_COAT_ALLOWANCE) / 2.0
                svg_lines.append(f'      <circle cx="{px:.3f}" cy="{py:.3f}" r="{cut_paracord_radius:.3f}" fill="black" />')

    # 7. Dedicated Corner Mounting Points
    if include_mounts:
        # Dynamically push mount holes inward if the chamfer gets very deep
        mount_offset = max(margin / 1.5, (corner_chamfer / 2.2) + softness)
        corner_mounts = [
            (mount_offset, mount_offset),
            (panel_width - mount_offset, mount_offset),
            (mount_offset, panel_height - mount_offset),
            (panel_width - mount_offset, panel_height - mount_offset)
        ]
        for mx, my in corner_mounts:
            cut_mount_radius = (mount_hole_dia + POWDER_COAT_ALLOWANCE) / 2.0
            svg_lines.append(f'      <circle cx="{mx:.3f}" cy="{my:.3f}" r="{cut_mount_radius:.3f}" fill="black" />')

    svg_lines.append('    </mask>')
    svg_lines.append('  </defs>')

    # 8. Draw the final black panel polygon and apply the mask
    # We must match the stroke parameters from the mask so the final shape exactly aligns
    svg_lines.append(f'  <polygon points="{poly_points}" fill="#000000" stroke="#000000" stroke-width="{stroke_w}" stroke-linejoin="round" mask="url(#cutout-mask)" />')

    svg_lines.append('</svg>')

    # Save the file locally
    filename = "fixed_molle_panel.svg"
    with open(filename, "w") as f:
        f.write("\n".join(svg_lines))

    return "\n".join(svg_lines)

# --- GRADIO INTERFACE ---
with gr.Blocks(title="MOLLE Panel Generator") as demo:
    gr.Markdown("## Interactive MOLLE/PALS Panel Generator (With Powder Coating Allowance)")

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### Dimensions & Margins")
            width_slider = gr.Slider(minimum=100, maximum=800, value=300, label="Panel Width (mm)")
            height_slider = gr.Slider(minimum=100, maximum=800, value=400, label="Panel Height (mm)")
            margin_slider = gr.Slider(minimum=10, maximum=100, value=30, label="Margin (mm)")

            gr.Markdown("### Geometry")
            paracord_slider = gr.Slider(minimum=2, maximum=10, value=5.0, label="Target Paracord Hole Dia (mm)")
            mount_slider = gr.Slider(minimum=3, maximum=15, value=6.5, label="Target Mount Hole Dia (mm)")
            chamfer_slider = gr.Slider(minimum=0, maximum=60, value=25.0, label="Corner Chamfer Amount (mm)")
            softness_slider = gr.Slider(minimum=0, maximum=20, value=6.0, label="Corner Softness / Rounding (mm)")

            gr.Markdown("### Features")
            with gr.Row():
                handle_top = gr.Checkbox(value=True, label="Top Handle")
                handle_bottom = gr.Checkbox(value=False, label="Bottom Handle")
            with gr.Row():
                handle_left = gr.Checkbox(value=False, label="Left Handle")
                handle_right = gr.Checkbox(value=False, label="Right Handle")

            mounts_toggle = gr.Checkbox(value=True, label="Include Corner Mount Holes")

        with gr.Column(scale=2):
            # Output window for the SVG
            svg_output = gr.HTML(label="Preview")
            gr.Markdown("*Output is automatically saved locally as `fixed_molle_panel.svg` on every change. Dimensions automatically compensated +0.3mm for paint buildup.*")

    # Tie the inputs to the function
    inputs = [
        width_slider, height_slider, margin_slider,
        paracord_slider, mount_slider, chamfer_slider, softness_slider,
        handle_top, handle_bottom, handle_left, handle_right, mounts_toggle
    ]

    for element in inputs:
        element.change(fn=generate_molle_svg, inputs=inputs, outputs=svg_output)

    # Run once on startup to render initial state
    demo.load(fn=generate_molle_svg, inputs=inputs, outputs=svg_output)

if __name__ == "__main__":
    demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://02ab3a01151f325516.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
